In [1]:
import pandas as pd

df = pd.read_csv('DimPrimeSlots.csv', encoding='latin1')

# Preview the data
print(df.shape)
print(df.head())
df.dtypes


(22914, 127)
   ï»¿Timestamp  Message LocationId  LocationGroupId  LocationReferenceKey  \
0           NaN      NaN    G701001              NaN                   NaN   
1           NaN      NaN    G703003              NaN                   NaN   
2           NaN      NaN    G705005              NaN                   NaN   
3           NaN      NaN    G707007              NaN                   NaN   
4           NaN      NaN    G709009              NaN                   NaN   

  LocationBarcode LocationTypeId LocationMaskId DisplayLocation  CheckDigit  \
0      G700160546        STORAGE          PRIME           G7001       546.0   
1      G700398182        STORAGE          PRIME           G7003       182.0   
2      G700595056        STORAGE          PRIME           G7005        56.0   
3      G700797911        STORAGE          PRIME           G7007       911.0   
4      G700972185        STORAGE          PRIME           G7009       185.0   

   ...  CheckDigitTypeId  ConsumeOnLocate  

C:\Users\James\AppData\Local\Temp\ipykernel_30264\3851161358.py:3: DtypeWarning: Columns (2,5,6,7,8,16,26,27,33,36,42,43,46,48,49,50,66,68,69,70,71,72,73,74,75,76,84,91,92,95,96,99,100,102,106,117,118,122,123,124,125,126) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('DimPrimeSlots.csv', encoding='latin1')


ï»¿Timestamp               float64
Message                    float64
LocationId                  object
LocationGroupId            float64
LocationReferenceKey       float64
                            ...   
TrackCapacityByQuantity     object
TrackCapacityByWeight       object
TrackCapacityByVolume       object
FinalStagingLocation        object
AllowMixingBusinessUnit     object
Length: 127, dtype: object

In [2]:
#Selecting columns of interest

wanted = ['LocationId', 'DisplayLocation', 'Aisle', 'Bay', 'PickExecutionSequence', 'PositionSequence']
df_clean = df[wanted].copy()
df_clean.head()

,LocationId,DisplayLocation,Aisle,Bay,PickExecutionSequence,PositionSequence
0,G701001,G7001,G7,1.0,1,1.0
1,G703003,G7003,G7,3.0,2,3.0
2,G705005,G7005,G7,5.0,3,5.0
3,G707007,G7007,G7,7.0,4,7.0
4,G709009,G7009,G7,9.0,5,9.0


In [3]:
df_clean.isnull().sum()


LocationId               21550
DisplayLocation          21550
Aisle                    21550
Bay                      21550
PickExecutionSequence    21550
PositionSequence         21550
dtype: int64

In [4]:
print(f"Unique LocationIds: {df_clean.duplicated(subset='LocationId').sum()}")


Unique LocationIds: 21549


In [5]:
df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(' ', '_')
df_clean.dtypes


locationid                object
displaylocation           object
aisle                     object
bay                      float64
pickexecutionsequence     object
positionsequence         float64
dtype: object

In [ ]:
# Remove only these specific LocationIds due to no SEQUENCE (fake slots)
exclude = ['E298098', 'E398098', 'E299099', 'E399099']
df_clean = df_clean[~df_clean['locationid'].isin(exclude)]

print(f"Rows remaining: {df_clean.shape[0]}")


Rows remaining: 1360


In [8]:
# Check where the nulls are
print(df_clean[['bay', 'positionsequence', 'pickexecutionsequence']].isnull().sum())

# Drop or fill nulls, then convert
df_clean.dropna(subset=['bay', 'positionsequence', 'pickexecutionsequence'], inplace=True)

# convert
df_clean['bay'] = df_clean['bay'].astype(int)
df_clean['positionsequence'] = df_clean['positionsequence'].astype(int)
df_clean['pickexecutionsequence'] = df_clean['pickexecutionsequence'].astype(int)


bay                      0
positionsequence         0
pickexecutionsequence    0
dtype: int64


In [11]:
df_clean


,locationid,displaylocation,aisle,bay,pickexecutionsequence,positionsequence
0,G701001,G7001,G7,1,1,1
1,G703003,G7003,G7,3,2,3
2,G705005,G7005,G7,5,3,5
3,G707007,G7007,G7,7,4,7
4,G709009,G7009,G7,9,5,9
...,...,...,...,...,...,...
1355,H803003,H8003,H8,3,1502,3
1356,H804004,H8004,H8,4,1503,4
1357,H805005,H8005,H8,5,1504,5
1358,H806006,H8006,H8,6,1505,6


In [12]:
#export to csv
df_clean.to_csv('PrimeSlots_cleaned.csv', index=False)
